# Figure-Sankey

Hydrogen project development: flow, duration and composition charts

Run this R notebook in the directory containing `master_data.rds` and `cloglog_data_full.rds` from Data Preparation.

- **Panel a:** annual status flows without a subtitle or unchanged-project annotation.
- **Panel b:** development-stage flows with filled, discrete duration ridges. The overall duration ridge beneath Operational is the sum of Concept, Feasibility and FID/Construction stage-duration distributions.
- **Panels c and d:** 100%-stacked Active/Inactive shares by region and end use, with total project counts shown in proportionally sized bubbles on a common scale.
- **Supplementary:** the standalone map and CSV tables documenting project counts, project-year observations, transitions by origin stage, region and end use, transition routes, and aggregate overlapping end-use membership.

**Overall-duration calculation.** The empirical, uncapped stage-duration probability distributions are convolved (summed), assuming independent durations across the three stages. This is a constructed distribution for a pathway through all three stages; it is not a distribution restricted to projects with an observed complete Concept-to-Operational history. The existing stage sample is retained: positive-progression spells, including stage jumps, excluding left-truncated origins. The stage plots retain the `5+` display bin, but the overall calculation uses the original durations before display binning. Filled stepwise ridges preserve the discrete annual measurements without a continuous density estimate.

**Counting conventions.** A project-year is a distinct `reference`–`year` pair in the source panel. The annual flow plot may carry statuses forward for plotting; those added plotting rows do not increase the observed project-year count. Project counts by region/end use use the latest record per project as of 2026. Stage project counts refer to projects observed in each stage anywhere in the panel and can overlap. Stage transition counts refer to outgoing pre-operational progression or failure events; direct stage jumps remain in the tables even though they are omitted from the compact flow diagram. Region/end-use assignment follows the 2026 snapshot for all years. “Duplicate end use” means membership in two or more of the five grouped end-use categories used in panel d; each project is counted once per category, and the no-recorded-use group is excluded from that multiplicity count.

The source `.rds` files are required to produce the revised charts and numerical outputs. Stored outputs from the earlier notebook have been cleared.


In [ ]:
# Use the same project/data directory in standalone and master-run execution.
# With H2_PROJECT_DIR unset, run from the folder containing Data/ and the RDS files.
project_dir <- Sys.getenv("H2_PROJECT_DIR", unset = getwd())
project_dir <- normalizePath(project_dir, winslash = "/", mustWork = TRUE)
setwd(project_dir)


In [ ]:
# ==============================================================================
# 0. SETUP AND DATA
# ============================================================================== 

required_packages <- c(
  "dplyr",
  "tidyr",
  "purrr",
  "ggplot2",
  "ggalluvial",
  "ggsci",
  "scales",
  "patchwork",
  "tibble",
  "grid",
  "sf",
  "stringr",
  "rnaturalearth",
  "ggrepel"
)

missing_packages <- required_packages[
  !vapply(
    required_packages,
    requireNamespace,
    logical(1),
    quietly = TRUE
  )
]

if (length(missing_packages) > 0) {
  stop(
    paste0(
      "Missing packages: ",
      paste(missing_packages, collapse = ", "),
      ". Install them before running this notebook."
    )
  )
}

invisible(
  lapply(
    required_packages,
    library,
    character.only = TRUE
  )
)

sf::sf_use_s2(TRUE)

master_data_file <- "master_data.rds"
cloglog_data_file <- "cloglog_data_full.rds"

if (!file.exists(master_data_file)) {
  stop(
    paste0(
      "The input file '",
      master_data_file,
      "' was not found. Run the Data Preparation notebook first and ensure its outputs are in this working directory."
    )
  )
}

if (!file.exists(cloglog_data_file)) {
  stop(
    paste0(
      "The input file '",
      cloglog_data_file,
      "' was not found. Run the Data Preparation notebook first and ensure its outputs are in this working directory."
    )
  )
}

master_data <- readRDS(master_data_file)
cloglog_data <- readRDS(cloglog_data_file)

if (!file.exists("sample_count_audit.rds")) {
  stop("Run the updated Data Preparation notebook to create sample_count_audit.rds.")
}
sample_count_audit <- readRDS("sample_count_audit.rds")
audit_inputs <- c(master_data_file, cloglog_data_file)
if (!identical(unname(sample_count_audit$source_md5[audit_inputs]),
               unname(tools::md5sum(audit_inputs)))) {
  stop("Sample-count audit and RDS inputs do not match. Re-run Data Preparation.")
}

required_master_columns <- c(
  "reference",
  "year",
  "status",
  "left_truncated"
)

missing_master_columns <- setdiff(
  required_master_columns,
  names(master_data)
)

if (length(missing_master_columns) > 0) {
  stop(
    paste0(
      "Missing required master_data columns: ",
      paste(missing_master_columns, collapse = ", ")
    )
  )
}

required_cloglog_columns <- c(
  "reference",
  "year",
  "status",
  "prev_status",
  "prev_time_in_status_raw",
  "prev_left_truncated"
)

missing_cloglog_columns <- setdiff(
  required_cloglog_columns,
  names(cloglog_data)
)

if (length(missing_cloglog_columns) > 0) {
  stop(
    paste0(
      "Missing required cloglog_data columns: ",
      paste(missing_cloglog_columns, collapse = ", ")
    )
  )
}

# Stratum order follows the vertical stacking used by ggalluvial.
# Cancellation is placed first so it appears at the top of each annual axis.
status_levels <- c(
  "Cancellation",
  "Concept",
  "Feasibility study",
  "FID/Construction",
  "Operational",
  "Decommissioned"
)

# Legend and colour assignment retain the development-stage sequence.
status_legend_levels <- c(
  "Concept",
  "Feasibility study",
  "FID/Construction",
  "Operational",
  "Decommissioned",
  "Cancellation"
)

pre_operational_stages <- c(
  "Concept",
  "Feasibility",
  "FID"
)

normalise_stage <- function(x) {
  x <- tolower(as.character(x))

  dplyr::case_when(
    grepl("concept", x) ~ "Concept",
    grepl("feasib", x) ~ "Feasibility",
    grepl("fid", x) ~ "FID",
    grepl("operat", x) ~ "Operational",
    grepl("decomm", x) ~ "Decommissioned",
    grepl("cancel", x) ~ "Cancellation",
    TRUE ~ NA_character_
  )
}

# Full Sankey sample: start from the complete project-year panel saved as
# master_data.rds so that no observations are lost through cloglog preparation.
sankey_data <- master_data %>%
  mutate(
    reference = as.character(reference),
    year = suppressWarnings(as.integer(as.character(year))),
    status = as.character(status),
    left_truncated = tidyr::replace_na(
      as.numeric(left_truncated),
      0
    ),
    curr_state = case_when(
      status == "Concept" ~ 1L,
      status == "Feasibility study" ~ 2L,
      status == "FID/Construction" ~ 3L,
      status == "Operational" ~ 4L,
      status %in% c(
        "Decommissioned",
        "Cancellation"
      ) ~ 0L,
      TRUE ~ NA_integer_
    )
  ) %>%
  arrange(
    reference,
    year
  ) %>%
  group_by(reference) %>%
  mutate(
    prev_state = lag(curr_state),
    prev_status = lag(status),
    prev_left_truncated = lag(
      left_truncated,
      default = first(left_truncated)
    ),
    transition_year = year
  ) %>%
  ungroup() %>%
  mutate(
    progress = as.integer(
      !is.na(prev_state) &
        !is.na(curr_state) &
        prev_state %in% 1:3 &
        curr_state > prev_state
    ),
    failure = as.integer(
      !is.na(prev_state) &
        !is.na(curr_state) &
        prev_state %in% 1:3 &
        curr_state == 0
    )
  )

# The duration sample uses cloglog_data with an origin-spell left-truncation
# restriction in the duration section.
cloglog_data <- cloglog_data %>%
  mutate(
    reference = as.character(reference),
    year = suppressWarnings(as.integer(as.character(year))),
    prev_left_truncated = tidyr::replace_na(
      as.numeric(prev_left_truncated),
      0
    )
  )

sankey_sample_audit <- sankey_data %>%
  summarise(
    n_rows = n(),
    n_projects = n_distinct(reference, na.rm = TRUE),
    n_left_truncated_origin_rows = sum(
      prev_left_truncated == 1,
      na.rm = TRUE
    )
  )

print(sankey_sample_audit)

message(
  scales::comma(sankey_sample_audit$n_projects),
  " projects and ",
  scales::comma(sankey_sample_audit$n_rows),
  " project-year rows retained from master_data, including ",
  scales::comma(sankey_sample_audit$n_left_truncated_origin_rows),
  " rows originating in left-truncated spells. These rows remain in the flow panels but are excluded from duration calculations."
)

options(
  repr.plot.width = 11.7,
  repr.plot.height = 11.0,
  repr.plot.res = 300
)


In [ ]:
# ==============================================================================
# 1. FIGURE-SPECIFIC DATA CONSTRUCTION AND VALUE CHECKS
# ============================================================================== 


# ------------------------------------------------------------------------------
# 1.1 Pipeline transitions using the full Data Preparation sample
# ------------------------------------------------------------------------------

pipeline_transition_rows <- sankey_data %>%
  filter(
    !is.na(reference),
    !is.na(prev_status),
    !is.na(progress),
    !is.na(failure)
  ) %>%
  mutate(
    prev_stage = normalise_stage(prev_status),
    observed_destination_stage = normalise_stage(status),
    transition_year = dplyr::coalesce(
      suppressWarnings(as.integer(as.character(transition_year))),
      year
    ),
    left_truncated_origin = prev_left_truncated == 1,

    # Failure nodes are origin-stage specific. Progression nodes use the
    # observed destination status so direct Concept -> FID or Concept ->
    # Operational transitions are not forced through an intermediate stage.
    next_stage = case_when(
      failure == 1 & prev_stage == "Concept" ~ "Failure_Concept",
      failure == 1 & prev_stage == "Feasibility" ~ "Failure_Feasibility",
      failure == 1 & prev_stage == "FID" ~ "Failure_FID",
      progress == 1 &
        observed_destination_stage %in% c(
          "Feasibility",
          "FID",
          "Operational"
        ) ~ observed_destination_stage,
      TRUE ~ NA_character_
    ),
    flow_type = case_when(
      grepl("^Failure_", next_stage) ~ "Failure",
      !is.na(next_stage) ~ "Progress",
      TRUE ~ NA_character_
    )
  ) %>%
  filter(
    prev_stage %in% pre_operational_stages,
    !is.na(next_stage),
    !is.na(flow_type)
  ) %>%
  distinct(
    reference,
    transition_year,
    prev_stage,
    next_stage,
    .keep_all = TRUE
  )

# Total flows include every transition. The left-truncated count is a subset of
# the total and identifies transitions originating from an incomplete initial
# spell.
pipeline_flow <- pipeline_transition_rows %>%
  group_by(
    prev_stage,
    next_stage,
    flow_type
  ) %>%
  summarise(
    n_total = n(),
    n_left_truncated = sum(
      left_truncated_origin,
      na.rm = TRUE
    ),
    n_non_left_truncated = n_total - n_left_truncated,
    .groups = "drop"
  ) %>%
  group_by(prev_stage) %>%
  mutate(
    stage_exit_total = sum(n_total),
    total_share = n_total / stage_exit_total,
    left_truncated_share = if_else(
      n_total > 0,
      n_left_truncated / n_total,
      0
    ),
flow_label = paste0(
  scales::comma(n_total),
  " transitions\n",
  "(full spells: ",
  scales::comma(n_non_left_truncated),
  ")"
)
  ) %>%
  ungroup()


pipeline_transition_rows %>%
  filter(
    !left_truncated_origin,
    is.na(country)
  ) %>%
  select(
    reference,
    transition_year,
    prev_stage,
    observed_destination_stage,
    flow_type,
    country
  )

# ------------------------------------------------------------------------------
# 1.2 Positive-progression duration distributions
# ------------------------------------------------------------------------------

progression_status_order <- c(
  "Concept",
  "Feasibility study",
  "FID/Construction",
  "Operational"
)

# Duration refers to the previous/origin status spell. Restricting on
# prev_left_truncated therefore excludes spells whose true start is not observed.
duration_project_year_sample <- cloglog_data %>%
  filter(
    prev_left_truncated == 0,
    !is.na(reference),
    !is.na(year),
    !is.na(status),
    !is.na(prev_status),
    !is.na(prev_time_in_status_raw)
  ) %>%
  mutate(
    status = as.character(status),
    prev_status = as.character(prev_status),
    duration_value = as.numeric(prev_time_in_status_raw),
    status_rank = match(status, progression_status_order),
    prev_status_rank = match(prev_status, progression_status_order),
    stage = normalise_stage(prev_status)
  ) %>%
  filter(
    is.finite(duration_value),
    duration_value >= 0,
    !is.na(prev_status_rank),
    stage %in% pre_operational_stages
  )

duration_data <- duration_project_year_sample %>%
  filter(
    !is.na(status_rank),
    status_rank > prev_status_rank
  ) %>%
  arrange(
    reference,
    prev_status_rank,
    status_rank,
    year
  ) %>%
  group_by(
    reference,
    prev_status,
    status
  ) %>%
  slice(1) %>%
  ungroup() %>%
  mutate(
    duration_bin = if_else(
      duration_value >= 5,
      5L,
      as.integer(floor(duration_value))
    )
  )

duration_distribution <- duration_data %>%
  count(
    stage,
    duration_bin,
    name = "n"
  ) %>%
  complete(
    stage = pre_operational_stages,
    duration_bin = 0:5,
    fill = list(n = 0)
  ) %>%
  group_by(stage) %>%
  mutate(
    total_n = sum(n),
    share = if_else(
      total_n > 0,
      n / total_n,
      0
    )
  ) %>%
  ungroup()

# Sum the uncapped empirical stage distributions. Do not add the display bins:
# a stage duration in the 5+ bin is not necessarily five years.
duration_stage_pmf <- duration_data %>%
  count(stage, duration_years = duration_value, name = "n") %>%
  group_by(stage) %>%
  mutate(probability = n / sum(n)) %>%
  ungroup()

convolve_duration_pmfs <- function(left, right) {
  tidyr::crossing(
    left %>% transmute(left_years = duration_years, left_p = probability),
    right %>% transmute(right_years = duration_years, right_p = probability)
  ) %>%
    transmute(
      duration_years = left_years + right_years,
      probability = left_p * right_p
    ) %>%
    group_by(duration_years) %>%
    summarise(probability = sum(probability), .groups = "drop") %>%
    arrange(duration_years)
}

duration_stage_samples <- duration_data %>%
  count(stage, name = "number_of_spells") %>%
  complete(stage = pre_operational_stages, fill = list(number_of_spells = 0L))

overall_duration_available <- all(duration_stage_samples$number_of_spells > 0)

if (overall_duration_available) {
  overall_duration_distribution <- purrr::reduce(
    lapply(pre_operational_stages, function(stage_name) {
      duration_stage_pmf %>%
        filter(stage == stage_name) %>%
        select(duration_years, probability)
    }),
    convolve_duration_pmfs
  )

  # Independent sums must retain unit probability and the sum of stage means.
  overall_duration_mean <- with(
    overall_duration_distribution,
    sum(duration_years * probability)
  )
  stopifnot(
    abs(sum(overall_duration_distribution$probability) - 1) < 1e-10,
    abs(overall_duration_mean - sum(
      duration_stage_pmf$duration_years * duration_stage_pmf$probability
    )) < 1e-8
  )
} else {
  overall_duration_distribution <- tibble(
    duration_years = numeric(), probability = numeric()
  )
  warning("Overall duration is unavailable: at least one stage has no eligible duration spells.")
}

hydrogen_snapshot_year <- 2026L

# Count source project-years, without the rows inserted for alluvial plotting.
flow_project_year_base <- sankey_data %>%
  filter(!is.na(reference), reference != "", !is.na(year)) %>%
  distinct(reference, year, .keep_all = TRUE)

flow_project_year_n <- nrow(flow_project_year_base)

# Count reconciliation for Figure 1 and Supplementary Tables 1-2.
# This is an audit: the existing plotting and estimation samples are preserved.
panel_counts <- sample_count_audit$audits$panel$summary
stopifnot(
  panel_counts$total_rows == nrow(sankey_data),
  panel_counts$distinct_valid_project_years == flow_project_year_n
)
fmt_count <- function(x) scales::comma(x, accuracy = 1)
sample_count_reconciliation_note <- paste0(
  "Figure 1 and Supplementary Tables 1-2 count distinct project-vintage pairs in the reconstructed panel. ",
  "The ", fmt_count(panel_counts$total_rows), " panel rows comprise ",
  fmt_count(panel_counts$missing_reference_rows), " rows with missing project identifiers, ",
  fmt_count(panel_counts$empty_reference_rows), " rows with empty project identifiers, ",
  fmt_count(panel_counts$missing_year_rows_with_valid_reference),
  " further rows with missing/invalid years, ",
  fmt_count(panel_counts$extra_duplicate_rows), " additional rows for repeated valid project-vintage pairs, and ",
  fmt_count(panel_counts$distinct_valid_project_years), " distinct valid project-vintage pairs. ",
  "Invalid identifiers/years are excluded from these counts; repeated valid pairs are counted once. ",
  "Of the counted pairs, ", fmt_count(panel_counts$project_years_with_observed_record),
  " contain an observed source record and ", fmt_count(panel_counts$project_years_only_reconstructed),
  " are present only through panel completion. Completed rows therefore are not additional observed database records."
)
if (panel_counts$duplicate_keys > 0L) {
  sample_count_reconciliation_note <- paste(
    sample_count_reconciliation_note,
    paste0("There are ", fmt_count(panel_counts$duplicate_keys),
      " repeated valid keys, of which ", fmt_count(panel_counts$conflicting_duplicate_keys),
      " have differing attributes; the detailed duplicate audit identifies the rows and fields for review.")
  )
}
if (panel_counts$whitespace_only_reference_rows > 0L) {
  sample_count_reconciliation_note <- paste(
    sample_count_reconciliation_note,
    paste0("REVIEW: ", fmt_count(panel_counts$whitespace_only_reference_rows),
      " whitespace-only identifiers remain under the existing empty-string counting rule.")
  )
}
print(sample_count_audit$summary, n = Inf, width = Inf)
message(sample_count_reconciliation_note)


flow_stage_project_year_counts <- flow_project_year_base %>%
  mutate(stage = normalise_stage(status)) %>%
  count(stage, name = "number_of_project_year_observations")

projects_2026_n <- sankey_data %>%
  filter(
    !is.na(reference),
    reference != "",
    !is.na(year),
    year <= hydrogen_snapshot_year
  ) %>%
  summarise(
    projects = n_distinct(
      reference,
      na.rm = TRUE
    )
  ) %>%
  pull(projects)

total_transition_n <- nrow(pipeline_transition_rows)

figure_subtitle_label <- paste0(
  scales::comma(projects_2026_n, accuracy = 1),
  " projects; ",
  scales::comma(flow_project_year_n, accuracy = 1),
  " project-year observations; ",
  scales::comma(total_transition_n, accuracy = 1),
  " transitions (positive progression + failure)"
)


# ------------------------------------------------------------------------------
# 1.3 Annual status data and no-change diagnostic using all observations
# ------------------------------------------------------------------------------

annual_status <- sankey_data %>%
  filter(
    !is.na(reference),
    !is.na(year),
    !is.na(status)
  ) %>%
  transmute(
    reference,
    year,
    status = as.character(status)
  ) %>%
  distinct(
    reference,
    year,
    .keep_all = TRUE
  ) %>%
  filter(
    status %in% status_levels
  )

year_levels <- annual_status %>%
  distinct(year) %>%
  arrange(year) %>%
  pull(year)

annual_status_complete <- annual_status %>%
  mutate(
    year = factor(
      year,
      levels = year_levels
    ),
    status = factor(
      status,
      levels = status_levels
    )
  ) %>%
  complete(
    reference,
    year = factor(
      year_levels,
      levels = year_levels
    ),
    fill = list(status = NA)
  ) %>%
  group_by(reference) %>%
  fill(
    status,
    .direction = "down"
  ) %>%
  ungroup() %>%
  filter(
    !is.na(status)
  ) %>%
  arrange(
    reference,
    year
  ) %>%
  group_by(reference) %>%
  mutate(
    previous_status = lag(status),
    no_stage_change = !is.na(previous_status) & status == previous_status
  ) %>%
  ungroup()

# Plotting-only pairs are counted explicitly, rather than inferred from row totals.
plotting_only_project_years <- annual_status_complete %>%
  distinct(reference, year) %>%
  mutate(year = suppressWarnings(as.integer(as.character(year)) )) %>%
  anti_join(flow_project_year_base %>% select(reference, year), by = c("reference", "year"))
sample_count_plotting_note <- paste0(
  "Annual status plotting adds ", fmt_count(nrow(plotting_only_project_years)),
  " project-vintage pairs beyond the counted source panel; these are excluded from Figure 1 and ST1-2 sample totals."
)
utils::write.csv(plotting_only_project_years, "sample_count_plotting_only_pairs.csv", row.names = FALSE)
message(sample_count_plotting_note)

no_change_n <- annual_status_complete %>%
  summarise(
    n = sum(no_stage_change, na.rm = TRUE)
  ) %>%
  pull(n)

no_change_projects_n <- annual_status_complete %>%
  filter(no_stage_change) %>%
  summarise(
    n = n_distinct(reference)
  ) %>%
  pull(n)



# ------------------------------------------------------------------------------
# 1.4 Explicit checks
# ------------------------------------------------------------------------------

pipeline_checks <- pipeline_flow %>%
  group_by(prev_stage) %>%
  summarise(
    counted_total_exits = sum(n_total),
    stored_total_exits = first(stage_exit_total),
    counted_left_truncated = sum(n_left_truncated),
    counted_non_left_truncated = sum(n_non_left_truncated),
    total_share_sum = sum(total_share),
    .groups = "drop"
  )

duration_checks <- duration_distribution %>%
  group_by(stage) %>%
  summarise(
    n_spells = first(total_n),
    counted_spells = sum(n),
    share_sum = sum(share),
    .groups = "drop"
  )

stopifnot(
  all(pipeline_checks$counted_total_exits == pipeline_checks$stored_total_exits),
  all(
    pipeline_checks$counted_total_exits ==
      pipeline_checks$counted_left_truncated +
      pipeline_checks$counted_non_left_truncated
  ),
  all(abs(pipeline_checks$total_share_sum - 1) < 1e-10),
  all(duration_checks$n_spells == duration_checks$counted_spells),
  all(abs(duration_checks$share_sum - if_else(duration_checks$n_spells > 0, 1, 0)) < 1e-10),
  all(duration_data$prev_left_truncated == 0)
)

pipeline_flow
duration_distribution
pipeline_checks
duration_checks

message(
  "Duration sample: ",
  scales::comma(n_distinct(duration_data$reference)),
  " projects and ",
  scales::comma(nrow(duration_data)),
  " positive progression spells with non-left-truncated origins."
)

message(
  "No-change diagnostic: ",
  scales::comma(no_change_n),
  " unchanged project-year transitions across ",
  scales::comma(no_change_projects_n),
  " projects."
)

print(duration_stage_samples)
print(overall_duration_distribution, n = Inf)
message(
  "Overall duration = Concept + Feasibility + FID/Construction; ",
  "convolution of the uncapped empirical stage distributions assumes independent stage durations."
)


In [ ]:

pipeline_transition_rows %>%
  filter(
    !left_truncated_origin,
    is.na(country)
  ) %>%
  select(
    reference,
    transition_year,
    prev_stage,
    observed_destination_stage,
    flow_type,
    country
  )

In [ ]:
# ==============================================================================
# 2. PANEL B: COMPACT PIPELINE WITH FILLED DISCRETE DURATION RIDGES
# ==============================================================================

stage_positions <- tibble::tribble(
  ~stage,        ~stage_label,          ~x,   ~box_width,
  "Concept",     "Concept",              1.00, 1.22,
  "Feasibility", "Feasibility",          3.30, 1.42,
  "FID",         "FID/Construction",     5.60, 1.72,
  "Operational", "Operational",          8.00, 1.42
)

stage_y <- 1.28
failure_y <- 0.42
stage_box_height <- 0.42
failure_box_width <- 1.14
failure_box_height <- 0.36

stage_nodes <- stage_positions %>%
  mutate(
    node_label = stage_label,
    y = stage_y,
    box_height = stage_box_height,
    node_type = if_else(
      stage == "Operational",
      "Outcome",
      "Stage"
    )
  )

failure_nodes <- stage_positions %>%
  filter(stage %in% pre_operational_stages) %>%
  transmute(
    stage,
    node = paste0("Failure_", stage),
    x,
    y = failure_y,
    box_width = failure_box_width,
    box_height = failure_box_height
  )

# Direct stage-jumping transitions remain in pipeline_flow but are deliberately
# omitted from the diagram. Only adjacent progression flows are plotted here.
adjacent_progress_edges <- pipeline_flow %>%
  filter(
    flow_type == "Progress",
    (
      prev_stage == "Concept" &
        next_stage == "Feasibility"
    ) |
      (
        prev_stage == "Feasibility" &
          next_stage == "FID"
      ) |
      (
        prev_stage == "FID" &
          next_stage == "Operational"
      )
  ) %>%
  left_join(
    stage_nodes %>%
      select(
        prev_stage = stage,
        source_x = x,
        source_y = y,
        source_width = box_width,
        source_height = box_height
      ),
    by = "prev_stage"
  ) %>%
  left_join(
    stage_nodes %>%
      select(
        next_stage = stage,
        target_x = x,
        target_y = y,
        target_width = box_width,
        target_height = box_height
      ),
    by = "next_stage"
  ) %>%
  mutate(
    x_start = source_x + source_width / 2,
    y_start = source_y,
    x_finish = target_x - target_width / 2,
    y_finish = target_y,
    label_x = (x_start + x_finish) / 2,
    label_y = stage_y + 0.42,
    compact_flow_label = paste0(
      scales::comma(n_total),
      " transitions\n(",
      scales::comma(n_non_left_truncated),
      " full spells)"
    )
  )

failure_edges <- pipeline_flow %>%
  filter(flow_type == "Failure") %>%
  left_join(
    stage_nodes %>%
      select(
        prev_stage = stage,
        source_x = x,
        source_y = y,
        source_width = box_width,
        source_height = box_height
      ),
    by = "prev_stage"
  ) %>%
  left_join(
    failure_nodes %>%
      select(
        next_stage = node,
        target_x = x,
        target_y = y,
        target_width = box_width,
        target_height = box_height
      ),
    by = "next_stage"
  ) %>%
  mutate(
    x_start = source_x,
    y_start = source_y - source_height / 2,
    x_finish = target_x,
    y_finish = target_y + target_height / 2,
    label_x = x_start + 0.50,
    label_y = (y_start + y_finish) / 2,
    compact_flow_label = paste0(
      scales::comma(n_total),
      " transitions\n(",
      scales::comma(n_non_left_truncated),
      " full spells)"
    )
  )

# Filled stepwise profiles keep the annual duration bins discrete. Ridge height
# is a probability share on a common vertical scale; the overall ridge uses its
# own explicitly labelled year range because it sums three stage durations.
duration_profile_width <- 1.52
duration_profile_ymin <- -0.72
duration_profile_ymax <- 0.10

overall_duration_max_year <- if (overall_duration_available) {
  max(1L, as.integer(floor(max(overall_duration_distribution$duration_years))))
} else {
  1L
}

overall_duration_plot_data <- if (overall_duration_available) {
  overall_duration_distribution %>%
    mutate(duration_bin = as.integer(floor(duration_years))) %>%
    group_by(duration_bin) %>%
    summarise(share = sum(probability), .groups = "drop") %>%
    complete(
      duration_bin = seq.int(0L, overall_duration_max_year),
      fill = list(share = 0)
    ) %>%
    mutate(stage = "Overall", n = NA_integer_, total_n = NA_integer_)
} else {
  tibble(
    duration_bin = integer(), share = numeric(), stage = character(),
    n = integer(), total_n = integer()
  )
}

duration_profile_positions <- stage_positions %>%
  transmute(
    stage = if_else(stage == "Operational", "Overall", stage),
    stage_x = x,
    max_bin = if_else(stage == "Overall", overall_duration_max_year, 5L)
  )

duration_profile_data <- bind_rows(
  duration_distribution %>% select(stage, duration_bin, n, total_n, share),
  overall_duration_plot_data
) %>%
  left_join(duration_profile_positions, by = "stage")

duration_share_limit <- max(0.65, duration_profile_data$share, na.rm = TRUE)

duration_profile_data <- duration_profile_data %>%
  mutate(
    # Each year is a unit-width probability-mass bin before scaling to the panel.
    x_left = stage_x - duration_profile_width / 2 +
      duration_bin / (max_bin + 1) * duration_profile_width,
    x_right = stage_x - duration_profile_width / 2 +
      (duration_bin + 1) / (max_bin + 1) * duration_profile_width,
    x = (x_left + x_right) / 2,
    y = duration_profile_ymin + share / duration_share_limit *
      (duration_profile_ymax - duration_profile_ymin)
  ) %>%
  arrange(stage, duration_bin)

# One closed polygon per ridge: a flat baseline and a stepwise empirical top.
duration_ridge_polygons <- bind_rows(lapply(
  unique(duration_profile_data$stage),
  function(stage_name) {
    stage_data <- duration_profile_data %>%
      filter(stage == stage_name) %>% arrange(duration_bin)
    top_x <- as.vector(rbind(stage_data$x_left, stage_data$x_right))
    top_y <- rep(stage_data$y, each = 2)
    tibble(
      stage = stage_name,
      x = c(first(stage_data$x_left), top_x, last(stage_data$x_right)),
      y = c(duration_profile_ymin, top_y, duration_profile_ymin)
    )
  }
))

duration_reference_shares <- pretty(c(0, duration_share_limit), n = 2)
duration_reference_shares <- duration_reference_shares[
  duration_reference_shares >= 0 & duration_reference_shares <= duration_share_limit
]

duration_grid_data <- duration_profile_positions %>%
  tidyr::crossing(reference_share = duration_reference_shares) %>%
  mutate(
    x = stage_x - duration_profile_width / 2,
    xend = stage_x + duration_profile_width / 2,
    y = duration_profile_ymin + reference_share / duration_share_limit *
      (duration_profile_ymax - duration_profile_ymin),
    yend = y
  )

duration_y_axis_labels <- tibble(reference_share = duration_reference_shares) %>%
  mutate(
    x = stage_positions$x[stage_positions$stage == "Concept"] -
      duration_profile_width / 2 - 0.08,
    y = duration_profile_ymin + reference_share / duration_share_limit *
      (duration_profile_ymax - duration_profile_ymin),
    label = scales::percent(reference_share, accuracy = 1)
  )

overall_duration_breaks <- unique(as.integer(round(seq(
  0, overall_duration_max_year, length.out = min(4L, overall_duration_max_year + 1L)
))))

duration_x_axis_labels <- duration_profile_data %>%
  filter(stage != "Overall" | duration_bin %in% overall_duration_breaks) %>%
  transmute(
    stage, x,
    y = duration_profile_ymin - 0.075,
    label = if_else(
      stage != "Overall" & duration_bin == 5L,
      "5+", as.character(duration_bin)
    )
  )

overall_duration_note_label <- tibble(
  stage = "Overall",
  label = if (overall_duration_available) "Sum of stage durations" else "Insufficient stage data"
) %>%
  left_join(duration_profile_positions, by = "stage") %>%
  mutate(x = stage_x, y = duration_profile_ymin - 0.20)


flow_colours <- c(
  "Progress" = "#4E6F8F",
  "Failure" = "#A6534B"
)

duration_colour <- ggsci::pal_npg("nrc")(10)[4]

p_pipeline_duration <- ggplot() +
  geom_segment(
    data = adjacent_progress_edges,
    aes(
      x = x_start,
      y = y_start,
      xend = x_finish,
      yend = y_finish,
      colour = flow_type,
      linewidth = n_total
    ),
    lineend = "round",
    arrow = grid::arrow(
      type = "closed",
      length = grid::unit(
        2.8,
        "mm"
      )
    )
  ) +
  geom_segment(
    data = failure_edges,
    aes(
      x = x_start,
      y = y_start,
      xend = x_finish,
      yend = y_finish,
      colour = flow_type,
      linewidth = n_total
    ),
    lineend = "round",
    arrow = grid::arrow(
      type = "closed",
      length = grid::unit(
        2.8,
        "mm"
      )
    )
  ) +
  geom_rect(
    data = stage_nodes,
    aes(
      xmin = x - box_width / 2,
      xmax = x + box_width / 2,
      ymin = y - box_height / 2,
      ymax = y + box_height / 2
    ),
    fill = "white",
    colour = "grey20",
    linewidth = 0.3
  ) +
  geom_rect(
    data = failure_nodes,
    aes(
      xmin = x - box_width / 2,
      xmax = x + box_width / 2,
      ymin = y - box_height / 2,
      ymax = y + box_height / 2
    ),
    fill = "grey95",
    colour = "grey20",
    linewidth = 0.3
  ) +
  geom_text(
    data = stage_nodes,
    aes(
      x = x,
      y = y,
      label = node_label
    ),
    size = 3.35
  ) +
  geom_text(
    data = failure_nodes,
    aes(
      x = x,
      y = y,
      label = "Failure"
    ),
    size = 3.2
  ) +
  geom_label(
    data = adjacent_progress_edges,
    aes(
      x = label_x,
      y = label_y,
      label = compact_flow_label,
      colour = flow_type
    ),
    size = 2.45,
    label.size = 0.15,
    label.padding = grid::unit(
      0.08,
      "lines"
    ),
    fill = "white",
    show.legend = FALSE
  ) +
  geom_label(
    data = failure_edges,
    aes(
      x = label_x,
      y = label_y,
      label = compact_flow_label,
      colour = flow_type
    ),
    size = 2.4,
    label.size = 0.15,
    label.padding = grid::unit(
      0.08,
      "lines"
    ),
    fill = "white",
    show.legend = FALSE
  ) +
  geom_segment(
    data = duration_grid_data,
    aes(x = x, y = y, xend = xend, yend = yend),
    inherit.aes = FALSE,
    colour = "grey88", linewidth = 0.24
  ) +
  geom_polygon(
    data = duration_ridge_polygons,
    aes(x = x, y = y, group = stage),
    inherit.aes = FALSE,
    fill = scales::alpha(duration_colour, 0.65),
    colour = duration_colour, linewidth = 0.4
  ) +
  geom_text(
    data = duration_y_axis_labels,
    aes(x = x, y = y, label = label),
    inherit.aes = FALSE, hjust = 1, colour = "grey35", size = 2.55
  ) +
  geom_text(
    data = duration_x_axis_labels,
    aes(x = x, y = y, label = label),
    inherit.aes = FALSE, colour = "grey35", size = 2.55
  ) +
  geom_text(
    data = overall_duration_note_label,
    aes(x = x, y = y, label = label),
    inherit.aes = FALSE, colour = "grey30", size = 2.7
  ) +
  annotate(
    "text", x = stage_positions$x[stage_positions$stage == "Operational"],
    y = duration_profile_ymax + 0.13,
    label = "Overall duration", colour = "grey25", size = 3
  ) +
  annotate(
    "text", x = -0.07,
    y = (duration_profile_ymin + duration_profile_ymax) / 2,
    label = "Share", angle = 90, colour = "grey35", size = 2.65
  ) +
  annotate(
    "text", x = 4.50, y = duration_profile_ymin - 0.32,
    label = "Duration to progression (years)", colour = "grey30", size = 2.8
  ) +
  scale_colour_manual(
    values = flow_colours,
    guide = "none"
  ) +
  scale_linewidth_continuous(
    range = c(
      0.42,
      1.55
    ),
    guide = "none"
  ) +
  coord_cartesian(
    xlim = c(
      -0.12,
      8.95
    ),
    ylim = c(
      -1.20,
      2.05
    ),
    clip = "off"
  ) +
  labs(
    tag = "b",
    title = "Hydrogen project development pipeline and stage durations",
    subtitle = figure_subtitle_label
  ) +
  theme_void(
    base_size = 10.5
  ) +
  theme(
    plot.tag = element_text(
      face = "plain",
      size = 11.5
    ),
    plot.tag.position = "topleft",
    plot.title = element_text(
      hjust = 0,
      face = "plain",
      size = 11,
      margin = margin(
        b = 2
      )
    ),
    plot.subtitle = element_text(
      hjust = 0,
      size = 8.5,
      margin = margin(
        b = 3
      )
    ),
    plot.margin = margin(
      8,
      8,
      5,
      2
    )
  )

p_pipeline_duration


In [ ]:
# ------------------------------------------------------------------------------
# Duration quantiles by stage and overall
# ------------------------------------------------------------------------------

duration_probabilities <- c(
  0.01,
  0.25,
  0.50,
  0.75,
  0.99
)

# Stage-specific empirical quantiles
duration_stage_quantiles <- duration_data %>%
  group_by(stage) %>%
  summarise(
    p1 = quantile(duration_value, 0.01, na.rm = TRUE),
    p25 = quantile(duration_value, 0.25, na.rm = TRUE),
    p50 = quantile(duration_value, 0.50, na.rm = TRUE),
    p75 = quantile(duration_value, 0.75, na.rm = TRUE),
    p99 = quantile(duration_value, 0.99, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    stage = recode(
      stage,
      "FID" = "FID/Construction"
    )
  )

# Probability-weighted quantile helper for overall convolved distribution
weighted_duration_quantile <- function(
    duration_years,
    probability,
    probs
) {
  ordered <- order(duration_years)

  duration_years <- duration_years[ordered]
  probability <- probability[ordered]

  cumulative_probability <- cumsum(probability)

  vapply(
    probs,
    function(p) {
      duration_years[
        which(cumulative_probability >= p)[1]
      ]
    },
    numeric(1)
  )
}

overall_duration_quantiles <- if (overall_duration_available) {
  overall_quantiles <- weighted_duration_quantile(
    duration_years = overall_duration_distribution$duration_years,
    probability = overall_duration_distribution$probability,
    probs = duration_probabilities
  )

  tibble(
    stage = "Overall",
    p1 = overall_quantiles[1],
    p25 = overall_quantiles[2],
    p50 = overall_quantiles[3],
    p75 = overall_quantiles[4],
    p99 = overall_quantiles[5]
  )
} else {
  tibble(
    stage = "Overall",
    p1 = NA_real_,
    p25 = NA_real_,
    p50 = NA_real_,
    p75 = NA_real_,
    p99 = NA_real_
  )
}

duration_quantile_summary <- bind_rows(
  duration_stage_quantiles,
  overall_duration_quantiles
) %>%
  mutate(
    stage = factor(
      stage,
      levels = c(
        "Concept",
        "Feasibility",
        "FID/Construction",
        "Overall"
      )
    )
  ) %>%
  arrange(stage)

print(
  duration_quantile_summary,
  n = Inf
)


In [ ]:
# ==============================================================================
# 3. PANEL A: ANNUAL STATUS TRANSITIONS
# ==============================================================================

status_cols <- ggsci::pal_npg()(
  length(status_legend_levels)
)
names(status_cols) <- status_legend_levels

status_sankey <- annual_status_complete %>%
  dplyr::group_by(reference) %>%
  dplyr::arrange(
    year,
    .by_group = TRUE
  ) %>%
  mutate(
    alluvium = reference,
    value = 1,
    next_status_raw = dplyr::lead(
      status
    ),
    transition_type = case_when(
      !is.na(next_status_raw) & status != next_status_raw ~ "Status change",
      !is.na(next_status_raw) & status == next_status_raw ~ "No change",
      TRUE ~ NA_character_
    ),
    next_status = if_else(
      is.na(next_status_raw),
      as.character(status),
      as.character(next_status_raw)
    ),
    next_status = factor(
      next_status,
      levels = status_levels
    ),
    transition_type = factor(
      transition_type,
      levels = c(
        "No change",
        "Status change"
      )
    ),

    # The final observation for each project has no outgoing transition, so
    # transition_type is NA. A complete plotting variable keeps final-year lodes
    # available when ggalluvial pairs adjacent axes.
    transition_type_plot = factor(
      tidyr::replace_na(
        as.character(transition_type),
        "No change"
      ),
      levels = c(
        "No change",
        "Status change"
      )
    )
  ) %>%
  ungroup()

stopifnot(
  !anyNA(status_sankey$next_status),
  !anyNA(status_sankey$transition_type_plot)
)

p_status_sankey <- ggplot(
  status_sankey,
  aes(
    x = year,
    stratum = status,
    alluvium = alluvium,
    y = value,
    fill = next_status
  )
) +
  geom_flow(
    aes(
      alpha = transition_type_plot
    ),
    aes.flow = "forward",
    colour = NA,
    width = 0.20,
    show.legend = FALSE
  ) +
  geom_stratum(
    aes(
      fill = status
    ),
    width = 0.20,
    colour = "black",
    linewidth = 0.25,
    alpha = 1,
    show.legend = TRUE
  ) +
  scale_x_discrete(
    expand = c(
      0.025,
      0.025
    )
  ) +
  scale_y_continuous(
    labels = scales::label_number(
      big.mark = ",",
      accuracy = 1
    ),
    expand = expansion(
      mult = c(
        0,
        0.04
      )
    )
  ) +
  scale_fill_manual(
    name = "Status",
    values = status_cols,
    breaks = status_legend_levels,
    drop = FALSE
  ) +
  scale_alpha_manual(
    values = c(
      "No change" = 0.12,
      "Status change" = 0.75
    ),
    guide = "none",
    na.value = 0
  ) +
  labs(
    tag = "a",
    title = "Hydrogen project status transitions by year",
    x = NULL,
    y = "Number of projects"
  ) +
  theme_minimal(
    base_size = 10
  ) +
  theme(
    panel.grid = element_blank(),
    plot.tag = element_text(
      face = "plain",
      size = 11.5
    ),
    plot.tag.position = "topleft",
    axis.line.x.bottom = element_line(
      colour = "black",
      linewidth = 0.3
    ),
    axis.line.y.left = element_line(
      colour = "black",
      linewidth = 0.3
    ),
    axis.ticks = element_line(
      colour = "black",
      linewidth = 0.3
    ),
    plot.title = element_text(
      hjust = 0,
      face = "plain",
      size = 11,
      margin = margin(
        b = 4
      )
    ),
    axis.text = element_text(
      size = 9.5
    ),
    axis.title = element_text(
      size = 10
    ),
    legend.position = "bottom",
    legend.title = element_text(
      size = 9.5
    ),
    legend.text = element_text(
      size = 9
    ),
    legend.box = "horizontal",
    plot.margin = margin(
      4,
      4,
      4,
      1
    )
  )

p_status_sankey


## 2026 project snapshot and standalone Supplementary map

The following section constructs one project-level 2026 snapshot. It is shared by panels c and d and by `extended_data_figure_map`. The map distinguishes active and inactive projects, scales projects with available capacity by electrolyser capacity, plots missing-capacity projects as small points, and annotates each assigned region with its project share.


In [ ]:
# ==============================================================================
# 4. PANEL C: GLOBAL PROJECT MAP
# ==============================================================================


# ------------------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------------------

hydrogen_map_year <- 2026L

hydrogen_map_xlim <- c(
  -180,
  180
)

hydrogen_map_ylim <- c(
  -58,
  85
)

hydrogen_map_max_bubble_size <- 8

npg_cols <- ggsci::pal_npg()(10)

hydrogen_col_active <- npg_cols[4]
hydrogen_col_inactive <- npg_cols[1]

hydrogen_col_country_fill <- "grey94"
hydrogen_col_country_outline <- "white"
hydrogen_col_point_outline <- "grey15"

hydrogen_status_levels <- c(
  "Active",
  "Inactive"
)

hydrogen_status_colours <- c(
  "Active" = hydrogen_col_active,
  "Inactive" = hydrogen_col_inactive
)


# ------------------------------------------------------------------------------
# Map style
# ------------------------------------------------------------------------------

hydrogen_project_map_theme <- theme_minimal(
  base_size = 8.5
) +
  theme(
    panel.grid = element_blank(),

    panel.border = element_rect(
      colour = "black",
      fill = NA,
      linewidth = 0.3
    ),

    axis.title = element_blank(),
    axis.text = element_blank(),
    axis.ticks = element_blank(),

    plot.title = element_text(
      hjust = 0,
      face = "bold",
      size = 10,
      margin = margin(
        b = 4
      )
    ),

    plot.subtitle = element_blank(),

    legend.title = element_text(
      size = 7.5
    ),

    legend.text = element_text(
      size = 7.5
    ),

    legend.key.size = grid::unit(
      0.40,
      "cm"
    ),

    legend.spacing.y = grid::unit(
      0.06,
      "cm"
    ),

    legend.position = "right",

    plot.margin = margin(
      4,
      4,
      4,
      4
    )
  )


# ------------------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------------------

clean_map_character <- function(x) {
  x_clean <- stringr::str_squish(
    as.character(x)
  )

  x_clean[
    is.na(x_clean) |
      x_clean == "" |
      stringr::str_to_lower(x_clean) %in% c(
        "na",
        "n/a",
        "null",
        "none"
      )
  ] <- NA_character_

  x_clean
}


safe_map_numeric <- function(x) {
  suppressWarnings(
    as.numeric(
      as.character(x)
    )
  )
}


map_binary_flag <- function(x) {
  x_character <- stringr::str_to_lower(
    stringr::str_squish(
      as.character(x)
    )
  )

  x_numeric <- suppressWarnings(
    as.numeric(
      x_character
    )
  )

  case_when(
    is.na(x_character) |
      x_character == "" ~
      FALSE,

    !is.na(x_numeric) ~
      x_numeric == 1,

    x_character %in% c(
      "true",
      "t",
      "yes",
      "y"
    ) ~
      TRUE,

    TRUE ~
      FALSE
  )
}


extract_map_year <- function(x) {
  year_character <- stringr::str_extract(
    as.character(x),
    "(19|20)[0-9]{2}"
  )

  suppressWarnings(
    as.integer(
      year_character
    )
  )
}


safe_map_quantile <- function(
    x,
    probability
) {
  x_clean <- x[
    is.finite(x)
  ]

  if (length(x_clean) == 0) {
    return(
      NA_real_
    )
  }

  as.numeric(
    quantile(
      x_clean,
      probs = probability,
      na.rm = TRUE,
      names = FALSE
    )
  )
}


# ------------------------------------------------------------------------------
# Check required and optional variables
# ------------------------------------------------------------------------------

required_master_map_vars <- c(
  "reference",
  "year",
  "name",
  "country",
  "region",
  "latitude",
  "longitude",
  "cap.mwel"
)

missing_master_map_vars <- setdiff(
  required_master_map_vars,
  names(master_data)
)

if (length(missing_master_map_vars) > 0) {
  stop(
    paste0(
      "Missing variables in master_data: ",
      paste(
        missing_master_map_vars,
        collapse = ", "
      )
    )
  )
}

optional_master_map_vars <- c(
  "location",
  "technology",
  "technology.aggregate",
  "product",
  "date.online",
  "date.decommissioned",
  "cap_mwel_revision_value",
  "status",
  "status_original",
  "status_harmonized",
  "status_backward_adjusted",
  "status_adjusted",
  "status_forward_filled",
  "status_created_cancellation",
  "curr_state",
  "curr_absorbing_state",
  "failure",
  "terminal_transition",
  "transition_type",
  "observed_row",
  "any_enduse",
  "enduse_industrial",
  "enduse_chemicals",
  "enduse_transport",
  "enduse_power",
  "enduse_other",
  "enduse.refining",
  "enduse.ammonia",
  "enduse.methanol",
  "enduse.ironsteel",
  "enduse.otherind",
  "enduse.mobility",
  "enduse.power",
  "enduse.gridinj",
  "enduse.chp",
  "enduse.domesticheat",
  "enduse.biofuels",
  "enduse.synfuels",
  "enduse.ch4gridinj",
  "enduse.ch4mobility"
)

master_map_input <- master_data

missing_optional_master_map_vars <- setdiff(
  optional_master_map_vars,
  names(master_map_input)
)

if (length(missing_optional_master_map_vars) > 0) {
  for (optional_variable in missing_optional_master_map_vars) {
    master_map_input[[optional_variable]] <- NA
  }
}


# ------------------------------------------------------------------------------
# Prepare all project-year records through 2026
# ------------------------------------------------------------------------------

hydrogen_project_years_for_map <- master_map_input %>%
  mutate(
    reference = clean_map_character(reference),

    year = suppressWarnings(
      as.integer(
        as.character(year)
      )
    ),

    name = clean_map_character(name),
    country = clean_map_character(country),
    region = clean_map_character(region),
    location = clean_map_character(location),
    technology = clean_map_character(technology),
    technology.aggregate = clean_map_character(technology.aggregate),
    product = clean_map_character(product),

    latitude = safe_map_numeric(latitude),
    longitude = safe_map_numeric(longitude),

    cap_mwel_original = safe_map_numeric(cap.mwel),
    cap_mwel_revision = safe_map_numeric(cap_mwel_revision_value),

    status_map_original = clean_map_character(status),
    status_map_status_original = clean_map_character(status_original),
    status_map_harmonized = clean_map_character(status_harmonized),
    status_map_backward_adjusted = clean_map_character(status_backward_adjusted),
    status_map_adjusted = clean_map_character(status_adjusted),
    status_map_forward_filled = clean_map_character(status_forward_filled),
    status_map_curr_state = clean_map_character(curr_state),
    status_map_curr_absorbing_state = clean_map_character(curr_absorbing_state),
    status_map_transition_type = clean_map_character(transition_type),

    date_online_year = extract_map_year(date.online),
    date_decommissioned_year = extract_map_year(date.decommissioned),

    observed_row_flag = map_binary_flag(observed_row),
    failure_flag = map_binary_flag(failure),
    cancellation_created_flag = map_binary_flag(status_created_cancellation),
    absorbing_state_flag = map_binary_flag(curr_absorbing_state),
    terminal_transition_flag = map_binary_flag(terminal_transition)
  ) %>%
  filter(
    !is.na(reference),
    reference != "",
    !is.na(year),
    year <= hydrogen_map_year
  )

if (nrow(hydrogen_project_years_for_map) == 0) {
  stop(
    paste0(
      "No project-year records were found on or before ",
      hydrogen_map_year,
      "."
    )
  )
}


# ------------------------------------------------------------------------------
# Select one record per project as of 2026
# ------------------------------------------------------------------------------

hydrogen_projects_2026 <- hydrogen_project_years_for_map %>%
  mutate(
    status_information_priority = case_when(
      !is.na(status_map_forward_filled) ~ 7L,
      !is.na(status_map_adjusted) ~ 6L,
      !is.na(status_map_backward_adjusted) ~ 5L,
      !is.na(status_map_harmonized) ~ 4L,
      !is.na(status_map_curr_state) ~ 3L,
      !is.na(status_map_status_original) ~ 2L,
      !is.na(status_map_original) ~ 1L,
      TRUE ~ 0L
    )
  ) %>%
  arrange(
    reference,
    desc(year),
    desc(observed_row_flag),
    desc(status_information_priority)
  ) %>%
  group_by(reference) %>%
  slice_head(n = 1) %>%
  ungroup() %>%
  rename(snapshot_year = year)


# ------------------------------------------------------------------------------
# Derive current capacity and active/inactive status
# ------------------------------------------------------------------------------

hydrogen_projects_2026 <- hydrogen_projects_2026 %>%
  mutate(
    project_name = coalesce(
      name,
      reference
    ),

    technology_for_map = coalesce(
      technology.aggregate,
      technology,
      product,
      "Unspecified"
    ),

    capacity_mw = case_when(
      is.finite(cap_mwel_revision) &
        cap_mwel_revision > 0 ~
        cap_mwel_revision,

      is.finite(cap_mwel_original) &
        cap_mwel_original > 0 ~
        cap_mwel_original,

      TRUE ~
        NA_real_
    ),

    capacity_source = case_when(
      is.finite(cap_mwel_revision) &
        cap_mwel_revision > 0 ~
        "cap_mwel_revision_value",

      is.finite(cap_mwel_original) &
        cap_mwel_original > 0 ~
        "cap.mwel",

      TRUE ~
        "Missing or non-positive capacity"
    ),

    status_source = coalesce(
      status_map_forward_filled,
      status_map_adjusted,
      status_map_backward_adjusted,
      status_map_harmonized,
      status_map_curr_state,
      status_map_status_original,
      status_map_original,
      status_map_curr_absorbing_state,
      status_map_transition_type,
      "Unspecified"
    ),

    status_text_all = stringr::str_to_lower(
      stringr::str_squish(
        paste(
          replace_na(status_map_forward_filled, ""),
          replace_na(status_map_adjusted, ""),
          replace_na(status_map_backward_adjusted, ""),
          replace_na(status_map_harmonized, ""),
          replace_na(status_map_curr_state, ""),
          replace_na(status_map_curr_absorbing_state, ""),
          replace_na(status_map_status_original, ""),
          replace_na(status_map_original, ""),
          replace_na(status_map_transition_type, ""),
          sep = " | "
        )
      )
    ),

    cancellation_text_flag = stringr::str_detect(
      status_text_all,
      paste0(
        "cancel|",
        "abandon|",
        "shelv|",
        "withdraw|",
        "terminat|",
        "reject|",
        "fail|",
        "decommission|",
        "closed|",
        "closure|",
        "inactive|",
        "suspend|",
        "stopp"
      )
    ),

    decommissioned_by_2026_flag =
      !is.na(date_decommissioned_year) &
      date_decommissioned_year <= hydrogen_map_year,

    project_inactive_flag =
      cancellation_text_flag |
      failure_flag |
      cancellation_created_flag |
      absorbing_state_flag |
      decommissioned_by_2026_flag,

    project_status_2026 = if_else(
      project_inactive_flag,
      "Inactive",
      "Active"
    ),

    project_status_2026 = factor(
      project_status_2026,
      levels = hydrogen_status_levels
    ),

    enduse_industrial_flag =
      map_binary_flag(enduse_industrial) |
      map_binary_flag(enduse.refining) |
      map_binary_flag(enduse.ironsteel) |
      map_binary_flag(enduse.otherind),

    enduse_chemicals_flag =
      map_binary_flag(enduse_chemicals) |
      map_binary_flag(enduse.ammonia) |
      map_binary_flag(enduse.methanol),

    enduse_transport_flag =
      map_binary_flag(enduse_transport) |
      map_binary_flag(enduse.mobility) |
      map_binary_flag(enduse.ch4mobility) |
      map_binary_flag(enduse.synfuels) |
      map_binary_flag(enduse.biofuels),

    enduse_power_flag =
      map_binary_flag(enduse_power) |
      map_binary_flag(enduse.power),

    enduse_other_flag =
      map_binary_flag(enduse_other) |
      map_binary_flag(enduse.gridinj) |
      map_binary_flag(enduse.ch4gridinj) |
      map_binary_flag(enduse.domesticheat) |
      map_binary_flag(enduse.chp),

    any_enduse_flag =
      map_binary_flag(any_enduse) |
      enduse_industrial_flag |
      enduse_chemicals_flag |
      enduse_transport_flag |
      enduse_power_flag |
      enduse_other_flag,

    coordinate_status = case_when(
      is.na(latitude) |
        is.na(longitude) ~
        "Missing coordinates",

      !is.finite(latitude) |
        !is.finite(longitude) ~
        "Invalid coordinates",

      latitude == 0 &
        longitude == 0 ~
        "Missing coordinates",

      latitude < -90 |
        latitude > 90 |
        longitude < -180 |
        longitude > 180 ~
        "Invalid coordinates",

      TRUE ~
        "Available coordinates"
    ),

    capacity_status = case_when(
      !is.na(capacity_mw) &
        is.finite(capacity_mw) &
        capacity_mw > 0 ~
        "Available capacity",

      TRUE ~
        "Missing capacity"
    )
  ) %>%
  arrange(
    project_status_2026,
    country,
    project_name,
    reference
  )


stopifnot(
  n_distinct(
    hydrogen_projects_2026$reference,
    na.rm = TRUE
  ) == projects_2026_n
)


# ------------------------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------------------------

hydrogen_project_map_summary <- tibble(
  metric = c(
    "Projects in 2026 snapshot",
    "Active projects",
    "Inactive projects",
    "Projects with valid coordinates",
    "Projects without valid coordinates",
    "Projects with reported MW capacity",
    "Projects without reported MW capacity",
    "Projects whose latest record is before 2026"
  ),

  value = c(
    n_distinct(hydrogen_projects_2026$reference),

    hydrogen_projects_2026 %>%
      filter(project_status_2026 == "Active") %>%
      summarise(n = n_distinct(reference)) %>%
      pull(n),

    hydrogen_projects_2026 %>%
      filter(project_status_2026 == "Inactive") %>%
      summarise(n = n_distinct(reference)) %>%
      pull(n),

    hydrogen_projects_2026 %>%
      filter(coordinate_status == "Available coordinates") %>%
      summarise(n = n_distinct(reference)) %>%
      pull(n),

    hydrogen_projects_2026 %>%
      filter(coordinate_status != "Available coordinates") %>%
      summarise(n = n_distinct(reference)) %>%
      pull(n),

    hydrogen_projects_2026 %>%
      filter(capacity_status == "Available capacity") %>%
      summarise(n = n_distinct(reference)) %>%
      pull(n),

    hydrogen_projects_2026 %>%
      filter(capacity_status == "Missing capacity") %>%
      summarise(n = n_distinct(reference)) %>%
      pull(n),

    hydrogen_projects_2026 %>%
      filter(snapshot_year < hydrogen_map_year) %>%
      summarise(n = n_distinct(reference)) %>%
      pull(n)
  )
)

hydrogen_project_status_summary <- hydrogen_projects_2026 %>%
  count(
    project_status_2026,
    sort = TRUE,
    name = "projects"
  ) %>%
  mutate(
    share = projects / sum(projects)
  )

hydrogen_project_capacity_summary <- hydrogen_projects_2026 %>%
  summarise(
    projects_with_capacity = sum(
      is.finite(capacity_mw) & capacity_mw > 0,
      na.rm = TRUE
    ),
    minimum_mw = safe_map_quantile(capacity_mw, 0),
    p25_mw = safe_map_quantile(capacity_mw, 0.25),
    median_mw = safe_map_quantile(capacity_mw, 0.50),
    p75_mw = safe_map_quantile(capacity_mw, 0.75),
    p95_mw = safe_map_quantile(capacity_mw, 0.95),
    maximum_mw = safe_map_quantile(capacity_mw, 1),
    total_mw = sum(capacity_mw, na.rm = TRUE)
  )

print(
  hydrogen_project_map_summary,
  n = Inf
)

print(
  hydrogen_project_status_summary,
  n = Inf
)

print(
  hydrogen_project_capacity_summary,
  n = Inf
)


# ------------------------------------------------------------------------------
# Regional project shares and non-overlapping annotation positions
# ------------------------------------------------------------------------------

# Regional shares are calculated among projects assigned to a named region.
# Missing and explicitly unassigned categories are omitted from the denominator.
hydrogen_region_share_data <- hydrogen_projects_2026 %>%
  filter(
    !is.na(region),
    region != "",
    !stringr::str_to_lower(region) %in% c(
      "unassigned",
      "unknown",
      "not assigned"
    )
  ) %>%
  group_by(region) %>%
  summarise(
    projects = n_distinct(reference),
    .groups = "drop"
  ) %>%
  mutate(
    share = projects / sum(projects),
    region_share_label = paste0(
      stringr::str_wrap(
        region,
        width = 22
      ),
      "\n",
      scales::percent(
        share,
        accuracy = 1
      )
    )
  ) %>%
  arrange(desc(projects))

# Use the median project coordinate as the anchor for each regional label, but
# place the actual label text at fixed positions in open parts of the map so
# that annotations do not overlap with project bubbles.
hydrogen_region_label_data <- hydrogen_projects_2026 %>%
  filter(
    coordinate_status == "Available coordinates",
    region %in% hydrogen_region_share_data$region
  ) %>%
  group_by(region) %>%
  summarise(
    anchor_longitude = stats::median(
      longitude,
      na.rm = TRUE
    ),
    anchor_latitude = stats::median(
      latitude,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  left_join(
    hydrogen_region_share_data,
    by = "region"
  ) %>%
  mutate(
    label_longitude = case_when(
      region == "Europe" ~ -28,
      region == "Asia Pacific" ~ 150,
      region == "US + Canada" ~ -150,
      region == "Latin America" ~ -118,
      region == "Others" ~ 32,
      TRUE ~ pmax(
        hydrogen_map_xlim[1] + 8,
        pmin(
          hydrogen_map_xlim[2] - 8,
          anchor_longitude + case_when(
            anchor_longitude < -30 ~ -24,
            anchor_longitude > 75 ~ 22,
            TRUE ~ 18
          )
        )
      )
    ),
    label_latitude = case_when(
      region == "Europe" ~ 66,
      region == "Asia Pacific" ~ -8,
      region == "US + Canada" ~ 60,
      region == "Latin America" ~ -18,
      region == "Others" ~ -8,
      TRUE ~ pmax(
        hydrogen_map_ylim[1] + 5,
        pmin(
          hydrogen_map_ylim[2] - 5,
          anchor_latitude + case_when(
            anchor_latitude > 40 ~ 10,
            anchor_latitude < -5 ~ -8,
            TRUE ~ 8
          )
        )
      )
    ),
    label_hjust = case_when(
      label_longitude < anchor_longitude ~ 1,
      label_longitude > anchor_longitude ~ 0,
      TRUE ~ 0.5
    )
  ) %>%
  filter(
    is.finite(anchor_longitude),
    is.finite(anchor_latitude),
    is.finite(label_longitude),
    is.finite(label_latitude)
  )

print(
  hydrogen_region_share_data,
  n = Inf
)

stopifnot(
  abs(
    sum(hydrogen_region_share_data$share) - 1
  ) < 1e-8
)


# ------------------------------------------------------------------------------
# Convert valid projects to sf and prepare layers
# ------------------------------------------------------------------------------

hydrogen_projects_2026_sf <- hydrogen_projects_2026 %>%
  filter(
    coordinate_status == "Available coordinates"
  ) %>%
  st_as_sf(
    coords = c(
      "longitude",
      "latitude"
    ),
    crs = 4326,
    remove = FALSE
  )

hydrogen_projects_active_capacity_sf <-
  hydrogen_projects_2026_sf %>%
  filter(
    project_status_2026 == "Active",
    capacity_status == "Available capacity"
  ) %>%
  arrange(
    desc(capacity_mw)
  )

hydrogen_projects_inactive_capacity_sf <-
  hydrogen_projects_2026_sf %>%
  filter(
    project_status_2026 == "Inactive",
    capacity_status == "Available capacity"
  ) %>%
  arrange(
    desc(capacity_mw)
  )

hydrogen_projects_active_missing_capacity_sf <-
  hydrogen_projects_2026_sf %>%
  filter(
    project_status_2026 == "Active",
    capacity_status == "Missing capacity"
  )

hydrogen_projects_inactive_missing_capacity_sf <-
  hydrogen_projects_2026_sf %>%
  filter(
    project_status_2026 == "Inactive",
    capacity_status == "Missing capacity"
  )


# ------------------------------------------------------------------------------
# Prepare world geometry and construct the map
# ------------------------------------------------------------------------------

hydrogen_world_map_sf <- rnaturalearth::ne_countries(
  scale = "medium",
  returnclass = "sf"
) %>%
  st_make_valid() %>%
  st_transform(4326) %>%
  filter(
    admin != "Antarctica"
  )

p_hydrogen_projects_2026_map <- ggplot() +
  geom_sf(
    data = hydrogen_world_map_sf,
    fill = hydrogen_col_country_fill,
    colour = hydrogen_col_country_outline,
    linewidth = 0.15
  ) +

  geom_sf(
    data = hydrogen_projects_active_missing_capacity_sf,
    aes(
      fill = project_status_2026
    ),
    shape = 21,
    size = 0.9,
    colour = hydrogen_col_point_outline,
    stroke = 0.15,
    alpha = 0.85,
    show.legend = FALSE
  ) +

  geom_sf(
    data = hydrogen_projects_active_capacity_sf,
    aes(
      size = capacity_mw,
      fill = project_status_2026
    ),
    shape = 21,
    colour = hydrogen_col_point_outline,
    stroke = 0.18,
    alpha = 0.82
  ) +

  geom_sf(
    data = hydrogen_projects_inactive_missing_capacity_sf,
    aes(
      fill = project_status_2026
    ),
    shape = 21,
    size = 0.9,
    colour = hydrogen_col_point_outline,
    stroke = 0.15,
    alpha = 0.95,
    show.legend = FALSE
  ) +

  geom_sf(
    data = hydrogen_projects_inactive_capacity_sf,
    aes(
      size = capacity_mw,
      fill = project_status_2026
    ),
    shape = 21,
    colour = hydrogen_col_point_outline,
    stroke = 0.22,
    alpha = 0.93
  ) +

  geom_curve(
    data = hydrogen_region_label_data,
    aes(
      x = anchor_longitude,
      y = anchor_latitude,
      xend = label_longitude,
      yend = label_latitude
    ),
    inherit.aes = FALSE,
    curvature = 0.10,
    linewidth = 0.20,
    colour = scales::alpha(
      "grey45",
      0.60
    ),
    lineend = "round",
    arrow = NULL
  ) +

  geom_text(
    data = hydrogen_region_label_data,
    aes(
      x = label_longitude,
      y = label_latitude,
      label = region_share_label,
      hjust = label_hjust
    ),
    inherit.aes = FALSE,
    size = 2.35,
    fontface = "bold",
    colour = "grey25",
    lineheight = 0.95
  ) +

  scale_fill_manual(
    values = hydrogen_status_colours,
    breaks = hydrogen_status_levels,
    drop = FALSE,
    name = "Project status\nas of 2026"
  ) +

  scale_size_area(
    max_size = hydrogen_map_max_bubble_size,
    breaks = scales::breaks_pretty(n = 4),
    labels = scales::label_number(
      accuracy = 1,
      big.mark = ",",
      suffix = " MW"
    ),
    name = "Electrolyser capacity\n(MWel)"
  ) +

  coord_sf(
    xlim = hydrogen_map_xlim,
    ylim = hydrogen_map_ylim,
    expand = FALSE,
    datum = NA
  ) +

  labs(
    title = "Hydrogen projects by status, 2026"
  ) +

  hydrogen_project_map_theme +

  guides(
    fill = guide_legend(
      order = 1,
      override.aes = list(
        shape = 21,
        size = 3.8,
        colour = hydrogen_col_point_outline,
        stroke = 0.2,
        alpha = 1
      )
    ),

    size = guide_legend(
      order = 2,
      override.aes = list(
        shape = 21,
        fill = "grey90",
        colour = hydrogen_col_point_outline,
        stroke = 0.2,
        alpha = 1
      )
    )
  )



# The map is retained as a separate Supplementary Figure S1 than a panel of
# the main figure.
extended_data_figure_map <- p_hydrogen_projects_2026_map

extended_data_figure_map


In [ ]:
# ==============================================================================
# 5. PANELS C AND D: ACTIVE/INACTIVE COMPOSITION BY REGION AND END USE
# ==============================================================================

preferred_region_order <- c(
  "Europe",
  "US + Canada",
  "Latin America",
  "Asia Pacific",
  "Africa + Middle East",
  "Others"
)

hydrogen_region_status_data <- hydrogen_projects_2026 %>%
  transmute(
    reference,
    region = clean_map_character(region),
    project_status_2026 = as.character(project_status_2026)
  ) %>%
  filter(
    !is.na(region),
    region != "",
    !stringr::str_to_lower(region) %in% c(
      "unassigned",
      "unknown",
      "not assigned"
    )
  ) %>%
  distinct(
    reference,
    region,
    project_status_2026
  ) %>%
  count(
    region,
    project_status_2026,
    name = "projects"
  ) %>%
  tidyr::complete(
    region,
    project_status_2026 = hydrogen_status_levels,
    fill = list(
      projects = 0
    )
  ) %>%
  group_by(region) %>%
  mutate(
    total_projects = sum(projects),
    share = if_else(
      total_projects > 0,
      projects / total_projects,
      0
    )
  ) %>%
  ungroup()

available_region_levels <- hydrogen_region_status_data %>%
  distinct(region) %>%
  pull(region)

region_plot_order <- c(
  intersect(
    preferred_region_order,
    available_region_levels
  ),
  sort(
    setdiff(
      available_region_levels,
      preferred_region_order
    )
  )
)

hydrogen_region_status_data <- hydrogen_region_status_data %>%
  mutate(
    region = factor(
      region,
      levels = rev(region_plot_order)
    ),
    project_status_2026 = factor(
      project_status_2026,
      levels = hydrogen_status_levels
    )
  )

enduse_flag_variables <- c(
  "enduse_industrial_flag",
  "enduse_chemicals_flag",
  "enduse_transport_flag",
  "enduse_power_flag",
  "enduse_other_flag"
)

enduse_labels <- c(
  enduse_industrial_flag = "Industrial",
  enduse_chemicals_flag = "Chemicals",
  enduse_transport_flag = "Transport",
  enduse_power_flag = "Power",
  enduse_other_flag = "Other",
  no_recorded_enduse_flag = "No recorded end use"
)

preferred_enduse_order <- unname(enduse_labels)

hydrogen_enduse_status_long <- hydrogen_projects_2026 %>%
  select(
    reference,
    project_status_2026,
    all_of(enduse_flag_variables)
  ) %>%
  mutate(
    across(
      all_of(enduse_flag_variables),
      ~ tidyr::replace_na(
        as.logical(.x),
        FALSE
      )
    ),
    no_recorded_enduse_flag =
      rowSums(
        across(
          all_of(enduse_flag_variables),
          ~ as.integer(.x)
        ),
        na.rm = TRUE
      ) == 0
  ) %>%
  pivot_longer(
    cols = c(
      all_of(enduse_flag_variables),
      no_recorded_enduse_flag
    ),
    names_to = "enduse_variable",
    values_to = "recorded_enduse"
  ) %>%
  filter(recorded_enduse) %>%
  mutate(
    end_use = unname(
      enduse_labels[enduse_variable]
    ),
    project_status_2026 = as.character(project_status_2026)
  ) %>%
  distinct(
    reference,
    end_use,
    project_status_2026
  )

hydrogen_enduse_status_data <- hydrogen_enduse_status_long %>%
  count(
    end_use,
    project_status_2026,
    name = "projects"
  ) %>%
  tidyr::complete(
    end_use,
    project_status_2026 = hydrogen_status_levels,
    fill = list(
      projects = 0
    )
  ) %>%
  group_by(end_use) %>%
  mutate(
    total_projects = sum(projects),
    share = if_else(
      total_projects > 0,
      projects / total_projects,
      0
    )
  ) %>%
  ungroup()

available_enduse_levels <- hydrogen_enduse_status_data %>%
  distinct(end_use) %>%
  pull(end_use)

enduse_plot_order <- intersect(
  preferred_enduse_order,
  available_enduse_levels
)

hydrogen_enduse_status_data <- hydrogen_enduse_status_data %>%
  mutate(
    end_use = factor(
      end_use,
      levels = rev(enduse_plot_order)
    ),
    project_status_2026 = factor(
      project_status_2026,
      levels = hydrogen_status_levels
    )
  )

status_breakdown_checks <- bind_rows(
  hydrogen_region_status_data %>%
    group_by(region) %>%
    summarise(
      breakdown = "Region",
      share_sum = sum(share),
      .groups = "drop"
    ) %>%
    transmute(
      breakdown,
      category = as.character(region),
      share_sum
    ),
  hydrogen_enduse_status_data %>%
    group_by(end_use) %>%
    summarise(
      breakdown = "End use",
      share_sum = sum(share),
      .groups = "drop"
    ) %>%
    transmute(
      breakdown,
      category = as.character(end_use),
      share_sum
    )
)

stopifnot(
  all(
    abs(status_breakdown_checks$share_sum - 1) < 1e-10
  )
)

print(
  hydrogen_region_status_data,
  n = Inf
)

print(
  hydrogen_enduse_status_data,
  n = Inf
)

status_breakdown_count_max <- max(
  c(
    hydrogen_region_status_data$total_projects,
    hydrogen_enduse_status_data$total_projects,
    1
  ),
  na.rm = TRUE
)

make_status_breakdown_plot <- function(
    data,
    category_variable,
    panel_tag,
    panel_title,
    panel_subtitle = NULL
) {
  total_label_data <- data %>%
    group_by(
      .data[[category_variable]]
    ) %>%
    summarise(
      total_projects = first(total_projects),
      .groups = "drop"
    ) %>%
    mutate(
      bubble_y = 1.11,
      bubble_size = scales::rescale(
        sqrt(total_projects),
        to = c(7.0, 13.5)
      )
    )

  ggplot(
    data,
    aes(
      x = .data[[category_variable]],
      y = share,
      fill = project_status_2026
    )
  ) +
    geom_col(
      width = 0.70,
      colour = "white",
      linewidth = 0.30
    ) +
    geom_text(
      aes(
        label = if_else(
          share >= 0.07 & projects > 0,
          scales::percent(share, accuracy = 1),
          ""
        )
      ),
      position = position_stack(
        vjust = 0.5
      ),
      colour = "white",
      size = 3.05
    ) +
    geom_point(
      data = total_label_data,
      aes(
        x = .data[[category_variable]],
        y = bubble_y,
        size = bubble_size
      ),
      inherit.aes = FALSE,
      shape = 21,
      fill = "white",
      colour = "grey20",
      stroke = 0.45
    ) +
    geom_text(
      data = total_label_data,
      aes(
        x = .data[[category_variable]],
        y = bubble_y,
        label = scales::comma(total_projects, accuracy = 1)
      ),
      inherit.aes = FALSE,
      colour = "grey15",
      size = 2.55
    ) +
    scale_y_continuous(
      breaks = seq(
        0,
        1,
        by = 0.25
      ),
      labels = scales::label_percent(
        accuracy = 1
      ),
      limits = c(
        0,
        1.20
      ),
      expand = c(
        0,
        0
      )
    ) +
    scale_size_identity(
      guide = "none"
    ) +
    scale_fill_manual(
      values = hydrogen_status_colours,
      breaks = hydrogen_status_levels,
      drop = FALSE,
      name = "Project status as of 2026"
    ) +
    coord_flip(
      clip = "off"
    ) +
    labs(
      tag = panel_tag,
      title = panel_title,
      subtitle = panel_subtitle,
      x = NULL,
      y = "Share of projects"
    ) +
    theme_minimal(
      base_size = 10.8
    ) +
    theme(
      panel.grid.major.y = element_blank(),
      panel.grid.minor = element_blank(),
      panel.grid.major.x = element_line(
        colour = "grey88",
        linewidth = 0.25
      ),
      axis.line.x = element_line(
        colour = "black",
        linewidth = 0.35
      ),
      axis.ticks.x = element_line(
        colour = "black",
        linewidth = 0.35
      ),
      plot.tag = element_text(
        face = "plain",
        size = 12
      ),
      plot.tag.position = "topleft",
      plot.title = element_text(
        face = "plain",
        size = 11.2,
        margin = margin(
          b = 3
        )
      ),
      plot.subtitle = element_text(
        size = 8.8,
        colour = "grey35",
        margin = margin(
          b = 4
        )
      ),
      axis.text.x = element_text(
        size = 9.6
      ),
      axis.text.y = element_text(
        size = 9.6,
        hjust = 0,
        margin = margin(
          r = 8
        )
      ),
      axis.title = element_text(
        size = 10
      ),
      legend.position = "bottom",
      legend.direction = "horizontal",
      legend.title = element_text(
        size = 9.2
      ),
      legend.text = element_text(
        size = 8.8
      ),
      legend.key.height = grid::unit(
        0.45,
        "cm"
      ),
      legend.key.width = grid::unit(
        0.75,
        "cm"
      ),
      plot.margin = margin(
        5,
        10,
        5,
        5
      )
    )
}

p_region_status <- make_status_breakdown_plot(
  data = hydrogen_region_status_data,
  category_variable = "region",
  panel_tag = "c",
  panel_title = "Project status by region, 2026"
)

p_enduse_status <- make_status_breakdown_plot(
  data = hydrogen_enduse_status_data,
  category_variable = "end_use",
  panel_tag = "d",
  panel_title = "Project status by end use, 2026",
  panel_subtitle = "Projects may have multiple recorded end uses"
)

p_status_breakdowns <-
  (
    p_region_status +
      p_enduse_status +
      patchwork::plot_layout(
        guides = "collect"
      )
  ) &
  theme(
    legend.position = "bottom"
  )

p_status_breakdowns


## Supplementary tables and end-use overlap

The combined table reports projects, distinct source project-years, outgoing pre-operational progression/failure transitions and full-spell transitions overall and by stage, region and end use. Stage project membership can overlap, while project-years are assigned to the stage observed in that year and transitions to their origin stage. Region/end-use rows use the latest classification per project as of 2026. Region totals reconcile to the overall sample; end-use totals can exceed it.

The notebook prints and writes:

- `extended_data_table_project_sample.csv`: combined sample table.
- `extended_data_table_transitions_by_stage.csv`: stage breakdown, including progression and failure separately.
- `extended_data_table_transition_routes.csv`: individual origin/destination routes, including stage jumps.
- `extended_data_table_enduse_overlap_summary.csv`: number of projects in multiple plotted end-use categories and additional category memberships.
- `extended_data_table_enduse_multiplicity.csv`: projects and project-years by number of recorded end-use categories.
- `extended_data_table_duplicate_enduse_projects.csv`: project identifiers and their overlapping categories.
- `overall_duration_distribution.csv`: the summed stage-duration probability distribution, before display binning.

The overlap calculation uses distinct grouped categories, so a broad indicator and its underlying detailed indicator do not count as two uses within the same category. No-recorded-use projects have zero recorded end uses.


In [ ]:
# ==============================================================================
# 6. EXTENDED DATA: STAGES, REGIONS, END USES AND END-USE OVERLAP
# ==============================================================================

extended_table_stage_labels <- c(
  Concept = "Concept",
  Feasibility = "Feasibility",
  FID = "FID/Construction",
  Operational = "Operational",
  Cancellation = "Cancellation",
  Decommissioned = "Decommissioned",
  Unassigned = "Unassigned"
)

# Source project-years only: no extra rows inserted by the flow-chart completion.
extended_table_project_year_base <- flow_project_year_base %>%
  transmute(
    reference, year,
    stage = tidyr::replace_na(normalise_stage(status), "Unassigned")
  )

# Each observed outgoing pre-operational progression/failure event counts once.
# Stage jumps are included. Full spells exclude left-truncated origin spells.
extended_table_transition_base <- pipeline_transition_rows %>%
  transmute(
    transition_id = row_number(), reference, transition_year,
    prev_stage, next_stage, flow_type, left_truncated_origin
  )

extended_table_region_lookup <- hydrogen_projects_2026 %>%
  transmute(reference, category = clean_map_character(region)) %>%
  mutate(
    category = case_when(
      is.na(category) | category == "" ~ "Unassigned",
      stringr::str_to_lower(category) %in%
        c("unassigned", "unknown", "not assigned") ~ "Unassigned",
      TRUE ~ category
    )
  ) %>%
  distinct(reference, category)

extended_table_enduse_lookup <- hydrogen_enduse_status_long %>%
  transmute(reference, category = as.character(end_use)) %>%
  distinct(reference, category)

summarise_transition_counts <- function(data, group_variable) {
  data %>%
    group_by(.data[[group_variable]]) %>%
    summarise(
      number_of_observed_transitions = n(),
      number_of_progression_transitions = sum(flow_type == "Progress"),
      number_of_failure_transitions = sum(flow_type == "Failure"),
      number_of_observed_transitions_with_full_spell = sum(
        !left_truncated_origin, na.rm = TRUE
      ),
      .groups = "drop"
    )
}

summarise_extended_table_group <- function(group_lookup, category_type_label) {
  project_counts <- group_lookup %>%
    group_by(category) %>%
    summarise(number_of_projects = n_distinct(reference), .groups = "drop")

  project_year_counts <- extended_table_project_year_base %>%
    inner_join(group_lookup, by = "reference", relationship = "many-to-many") %>%
    group_by(category) %>%
    summarise(number_of_project_year_observations = n(), .groups = "drop")

  transition_counts <- extended_table_transition_base %>%
    inner_join(group_lookup, by = "reference", relationship = "many-to-many") %>%
    summarise_transition_counts("category")

  project_counts %>%
    left_join(project_year_counts, by = "category") %>%
    left_join(transition_counts, by = "category") %>%
    mutate(
      across(starts_with("number_of_"), ~ tidyr::replace_na(as.integer(.x), 0L)),
      category_type = category_type_label, .before = category
    )
}

extended_table_overall <- tibble(
  category_type = "Overall", category = "All projects",
  number_of_projects = n_distinct(hydrogen_projects_2026$reference),
  number_of_project_year_observations = nrow(extended_table_project_year_base),
  number_of_observed_transitions = nrow(extended_table_transition_base),
  number_of_progression_transitions = sum(extended_table_transition_base$flow_type == "Progress"),
  number_of_failure_transitions = sum(extended_table_transition_base$flow_type == "Failure"),
  number_of_observed_transitions_with_full_spell = sum(
    !extended_table_transition_base$left_truncated_origin, na.rm = TRUE
  )
)

# Stage membership varies over time. Count only the project-years actually
# observed in each stage, rather than assigning a project's entire history to it.
extended_table_stage_projects <- extended_table_project_year_base %>%
  group_by(stage) %>%
  summarise(
    number_of_projects = n_distinct(reference),
    number_of_project_year_observations = n(),
    .groups = "drop"
  )

extended_table_stage_transitions <- extended_table_transition_base %>%
  mutate(stage = prev_stage) %>%
  summarise_transition_counts("stage")

extended_table_stage_order <- c(
  "Concept", "Feasibility", "FID", "Operational", "Cancellation", "Decommissioned"
)
if (any(extended_table_project_year_base$stage == "Unassigned")) {
  extended_table_stage_order <- c(extended_table_stage_order, "Unassigned")
}

extended_table_by_stage <- tibble(stage = extended_table_stage_order) %>%
  left_join(extended_table_stage_projects, by = "stage") %>%
  left_join(extended_table_stage_transitions, by = "stage") %>%
  mutate(
    across(starts_with("number_of_"), ~ tidyr::replace_na(as.integer(.x), 0L)),
    category_type = "Stage",
    category = unname(extended_table_stage_labels[stage])
  ) %>%
  select(category_type, category, starts_with("number_of_"))

extended_table_by_region <- summarise_extended_table_group(
  extended_table_region_lookup, "Region"
)
extended_table_by_enduse <- summarise_extended_table_group(
  extended_table_enduse_lookup, "End use"
)

extended_data_table_project_sample <- bind_rows(
  extended_table_overall %>% mutate(category_type_order = 1L, category_order = 1L),
  extended_table_by_stage %>% mutate(category_type_order = 2L, category_order = row_number()),
  extended_table_by_region %>% mutate(
    category_type_order = 3L,
    category_order = match(category, c(preferred_region_order, "Unassigned"))
  ),
  extended_table_by_enduse %>% mutate(
    category_type_order = 4L, category_order = match(category, preferred_enduse_order)
  )
) %>%
  arrange(category_type_order, category_order, category) %>%
  select(-category_type_order, -category_order)

extended_data_table_transitions_by_stage <- extended_table_by_stage

extended_data_table_transition_routes <- extended_table_transition_base %>%
  group_by(prev_stage, next_stage, flow_type) %>%
  summarise(
    number_of_projects = n_distinct(reference),
    number_of_observed_transitions = n(),
    number_of_observed_transitions_with_full_spell = sum(
      !left_truncated_origin, na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  mutate(
    stage_order = match(prev_stage, pre_operational_stages),
    origin_stage = unname(extended_table_stage_labels[prev_stage]),
    destination_stage = if_else(
      grepl("^Failure_", next_stage), "Failure",
      unname(extended_table_stage_labels[next_stage])
    )
  ) %>%
  arrange(stage_order, flow_type, destination_stage) %>%
  select(origin_stage, destination_stage, flow_type, starts_with("number_of_"))

# Multi-category projects cause the overlap in panel d. Count distinct grouped
# uses, not repeated records or both a broad flag and its detailed sub-flags.
extended_table_recorded_enduse_counts <- extended_table_enduse_lookup %>%
  filter(category != "No recorded end use") %>%
  group_by(reference) %>%
  summarise(
    number_of_recorded_enduses = n_distinct(category),
    recorded_enduses = paste(sort(unique(category)), collapse = "; "),
    .groups = "drop"
  )

extended_table_project_year_counts <- extended_table_project_year_base %>%
  count(reference, name = "number_of_project_year_observations")

extended_table_enduse_project_counts <- hydrogen_projects_2026 %>%
  distinct(reference) %>%
  left_join(extended_table_recorded_enduse_counts, by = "reference") %>%
  left_join(extended_table_project_year_counts, by = "reference") %>%
  mutate(
    number_of_recorded_enduses = tidyr::replace_na(number_of_recorded_enduses, 0L),
    recorded_enduses = tidyr::replace_na(recorded_enduses, "No recorded end use"),
    number_of_project_year_observations = tidyr::replace_na(
      number_of_project_year_observations, 0L
    ),
    duplicate_enduse = number_of_recorded_enduses > 1L,
    additional_category_memberships = pmax(number_of_recorded_enduses - 1L, 0L)
  )

extended_data_table_enduse_overlap_summary <- extended_table_enduse_project_counts %>%
  summarise(
    number_of_projects = n(),
    number_of_projects_with_recorded_enduse = sum(number_of_recorded_enduses > 0L),
    number_of_projects_without_recorded_enduse = sum(number_of_recorded_enduses == 0L),
    number_of_projects_with_duplicate_enduse = sum(duplicate_enduse),
    share_of_projects_with_duplicate_enduse = if (n() > 0) mean(duplicate_enduse) else NA_real_,
    number_of_project_year_observations_for_duplicate_enduse_projects = sum(
      number_of_project_year_observations[duplicate_enduse]
    ),
    number_of_additional_enduse_category_memberships = sum(additional_category_memberships)
  )

extended_data_table_enduse_multiplicity <- extended_table_enduse_project_counts %>%
  group_by(number_of_recorded_enduses) %>%
  summarise(
    number_of_projects = n(),
    number_of_project_year_observations = sum(number_of_project_year_observations),
    .groups = "drop"
  ) %>%
  complete(
    number_of_recorded_enduses = 0:length(enduse_flag_variables),
    fill = list(number_of_projects = 0L, number_of_project_year_observations = 0L)
  ) %>%
  mutate(share_of_projects = number_of_projects / sum(number_of_projects))

extended_data_table_note <- paste(
  sample_count_reconciliation_note,
  sample_count_plotting_note,
  "Region/end-use project counts and classifications refer to the latest record per project as of 2026.",
  "Project-years are distinct source-panel reference-year pairs, excluding additional plotting-only carry-forward rows.",
  "Stage project counts refer to projects observed in that stage at any time; these can overlap.",
  "Project-years are assigned to their current stage, and transitions to their origin stage.",
  "Transitions are outgoing pre-operational progression/failure events, including direct stage jumps.",
  "Consequently Operational, Cancellation and Decommissioned have zero outgoing transitions in this table's scope.",
  "Full-spell transitions exclude left-truncated origin spells.",
  "Region groups are mutually exclusive; end-use groups overlap.",
  "Duplicate end use means two or more distinct grouped end-use categories; No recorded end use counts as zero."
)

# Reconciliation checks catch dropped groups and accidental join multiplication.
stopifnot(
  nrow(extended_table_region_lookup) == projects_2026_n,
  n_distinct(extended_table_region_lookup$reference) == nrow(extended_table_region_lookup),
  n_distinct(extended_table_enduse_lookup$reference) == projects_2026_n,
  sum(extended_table_by_region$number_of_projects) == extended_table_overall$number_of_projects,
  all(extended_data_table_project_sample$number_of_observed_transitions ==
    extended_data_table_project_sample$number_of_progression_transitions +
    extended_data_table_project_sample$number_of_failure_transitions),
  all(extended_data_table_project_sample$number_of_observed_transitions_with_full_spell <=
    extended_data_table_project_sample$number_of_observed_transitions),
  sum(extended_data_table_enduse_multiplicity$number_of_projects) == projects_2026_n,
  sum(extended_data_table_enduse_multiplicity$number_of_project_year_observations) == flow_project_year_n,
  sum(extended_table_by_enduse$number_of_projects) == projects_2026_n +
    extended_data_table_enduse_overlap_summary$number_of_additional_enduse_category_memberships,
  sum(extended_data_table_transition_routes$number_of_observed_transitions) ==
    nrow(extended_table_transition_base)
)

for (count_column in c(
  "number_of_project_year_observations", "number_of_observed_transitions",
  "number_of_progression_transitions", "number_of_failure_transitions",
  "number_of_observed_transitions_with_full_spell"
)) {
  stopifnot(
    sum(extended_table_by_region[[count_column]]) == extended_table_overall[[count_column]],
    sum(extended_table_by_stage[[count_column]]) == extended_table_overall[[count_column]]
  )
}

print(extended_data_table_project_sample, n = Inf, width = Inf)
print(extended_data_table_transitions_by_stage, n = Inf, width = Inf)
print(extended_data_table_transition_routes, n = Inf, width = Inf)
print(extended_data_table_enduse_overlap_summary, width = Inf)
print(extended_data_table_enduse_multiplicity, n = Inf, width = Inf)
# Caption-ready text is generated from this run's counts, never hard-coded.
figure_sample_caption_note <- paste(
  sample_count_reconciliation_note, sample_count_plotting_note,
  paste0("The figure covers ", fmt_count(projects_2026_n), " projects and ",
    fmt_count(total_transition_n), " outgoing pre-operational progression/failure transitions, including stage jumps. ",
    "Flows include left-truncated origin spells; duration summaries exclude them.")
)
sample_count_caption_notes <- c(
  "Figure 1 - sample-count caption note", figure_sample_caption_note, "",
  "Supplementary Tables 1-2 - sample-count/table note", extended_data_table_note
)
writeLines(sample_count_caption_notes, "Figure1_ST1_ST2_sample_count_caption_notes.txt", useBytes = TRUE)
message(extended_data_table_note)
message(
  "Projects with duplicate end use (two or more grouped categories): ",
  scales::comma(extended_data_table_enduse_overlap_summary$number_of_projects_with_duplicate_enduse),
  " of ", scales::comma(projects_2026_n), " (",
  scales::percent(extended_data_table_enduse_overlap_summary$share_of_projects_with_duplicate_enduse,
    accuracy = 0.1), ")."
)

extended_data_outputs <- list(
  extended_data_table_project_sample = extended_data_table_project_sample,
  extended_data_table_transitions_by_stage = extended_data_table_transitions_by_stage,
  extended_data_table_transition_routes = extended_data_table_transition_routes,
  extended_data_table_enduse_overlap_summary = extended_data_table_enduse_overlap_summary,
  extended_data_table_enduse_multiplicity = extended_data_table_enduse_multiplicity,
  overall_duration_distribution = overall_duration_distribution
)

for (output_name in names(extended_data_outputs)) {
  utils::write.csv(
    extended_data_outputs[[output_name]],
    file = paste0(output_name, ".csv"), row.names = FALSE, na = ""
  )
}


In [ ]:
# ==============================================================================
# 7. ASSEMBLE AND EXPORT THE MAIN FIGURE AND EXTENDED DATA MAP
# ==============================================================================

# Preserve the intrinsic left-side width of panels c/d so the region labels in
# panel c are fully included in the assembled figure. This changes horizontal
# spacing only; the exported figure height remains unchanged.
p_status_breakdowns_free_left <- p_status_breakdowns

p_panel_integrated_duration_flow <-
  patchwork::wrap_plots(
    p_status_sankey,
    p_pipeline_duration,
    p_status_breakdowns_free_left,
    ncol = 1,
    heights = c(
      1.25,
      1.45,
      1.15
    )
  )

p_panel_integrated_duration_flow

ggsave(
  filename = "figure_sankey.pdf",
  plot = p_panel_integrated_duration_flow,
  device = cairo_pdf,
  width = 210,
  height = 260,
  units = "mm",
  dpi = 400
)

extended_data_figure_map

ggsave(
  filename = "figure_S1_project_map.pdf",
  plot = extended_data_figure_map,
  device = cairo_pdf,
  width = 210,
  height = 115,
  units = "mm",
  dpi = 400
)


## Supplementary Figure S6: Capacity by end use

Capacity distributions use the 2026 project snapshot and the same grouped end-use definitions as panel d. Projects with multiple recorded end uses contribute to each applicable end-use category. Only projects with positive, finite capacity are included.


In [ ]:
# ==============================================================================
# 8. SUPPLEMENTARY FIGURE: PROJECT CAPACITY BY END USE
# ============================================================================== 

supplementary_capacity_by_enduse_data <- hydrogen_enduse_status_long %>%
  left_join(
    hydrogen_projects_2026 %>%
      select(reference, capacity_mw),
    by = "reference"
  ) %>%
  filter(
    is.finite(capacity_mw),
    capacity_mw > 0
  ) %>%
  mutate(
    end_use = factor(
      end_use,
      levels = rev(enduse_plot_order)
    )
  )

supplementary_capacity_by_enduse_summary <- supplementary_capacity_by_enduse_data %>%
  group_by(end_use) %>%
  summarise(
    number_of_projects = n_distinct(reference),
    p25_mw = quantile(capacity_mw, 0.25, na.rm = TRUE),
    median_mw = median(capacity_mw, na.rm = TRUE),
    p75_mw = quantile(capacity_mw, 0.75, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(end_use)

print(
  supplementary_capacity_by_enduse_summary,
  n = Inf
)

supplementary_capacity_by_enduse_labels <- setNames(
  paste0(
    as.character(
      supplementary_capacity_by_enduse_summary$end_use
    ),
    " (n = ",
    supplementary_capacity_by_enduse_summary$number_of_projects,
    ")"
  ),
  as.character(
    supplementary_capacity_by_enduse_summary$end_use
  )
)

# NPG / NRC palette
nrc_cols <- ggsci::pal_npg("nrc")(10)

supplementary_figure_capacity_by_enduse <- ggplot(
  supplementary_capacity_by_enduse_data,
  aes(
    x = end_use,
    y = capacity_mw
  )
) +
  geom_violin(
    width = 0.72,
    linewidth = 0.55,
    scale = "width",
    trim = FALSE,
    fill = nrc_cols[2],
    colour = "grey20"
  ) +
  geom_linerange(
    data = supplementary_capacity_by_enduse_summary,
    aes(
      x = end_use,
      ymin = p25_mw,
      ymax = p75_mw
    ),
    inherit.aes = FALSE,
    linewidth = 2.4,
    colour = "grey20",
    lineend = "round"
  ) +
  geom_point(
    data = supplementary_capacity_by_enduse_summary,
    aes(
      x = end_use,
      y = median_mw
    ),
    inherit.aes = FALSE,
    shape = 21,
    size = 2.3,
    stroke = 0.55,
    fill = "white",
    colour = "grey10"
  ) +
  scale_x_discrete(
    labels = supplementary_capacity_by_enduse_labels
  ) +
  scale_y_log10(
    breaks = scales::breaks_log(
      n = 6
    ),
    labels = scales::label_number(
      big.mark = ",",
      accuracy = 1
    )
  ) +
  coord_flip() +
  labs(
    x = NULL,
    y = "Project capacity (MW, log scale)"
  ) +
  theme_minimal(
    base_size = 12.5
  ) +
  theme(
    panel.grid.major.y = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_line(
      colour = "grey88",
      linewidth = 0.30
    ),
    axis.line.x = element_line(
      colour = "black",
      linewidth = 0.40
    ),
    axis.ticks.x = element_line(
      colour = "black",
      linewidth = 0.40
    ),
    axis.text.x = element_text(
      size = 11.5,
      colour = "black"
    ),
    axis.text.y = element_text(
      size = 11.5,
      colour = "black",
      hjust = 0,
      margin = margin(r = 8)
    ),
    axis.title.x = element_text(
      size = 12,
      margin = margin(t = 8)
    ),
    plot.margin = margin(
      7,
      10,
      7,
      12
    )
  )

supplementary_figure_capacity_by_enduse

ggsave(
  filename = "figure_S6_capacity_by_end_use.pdf",
  plot = supplementary_figure_capacity_by_enduse,
  device = cairo_pdf,
  width = 160,
  height = 105,
  units = "mm",
  dpi = 400
)
